# Explore a speed objective and NeRD design proposals

NeRD learns an approximation of trajectories produced by the analytical solver.
An optimizer uses its gradients to propose physical designs, then the solver
checks those proposals. This notebook explores the speed cost and compares
recorded predictions with their physical verification.

Open in JupyterLab with `.[analysis,notebooks]` installed and run the cells in
order. The training step at the end additionally uses `.[learning]` and a
checkout of the official NeRD implementation.


## 1. Follow the design optimization cycle

1. Choose a parameter domain and sample physical designs.
2. Simulate those designs to generate trajectories and physical checks.
3. Train NeRD on a training split; measure accuracy on separate designs.
4. Freeze its weights and differentiate the chosen cost with respect to design.
5. Simulate proposed designs at finer resolution and check physical constraints.
6. Use failed proposals to improve training coverage while retaining independent validation designs.

The current expanded experiment uses supported, zero-phase starts at fixed
200 V / 363 Hz. Its held-out accuracy checks fail, so its predictions remain
experimental. [The expanded study](../docs/expanded_design.md) records the domain,
results and current limits.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Open from the repository or tutorials directory."
os.chdir(ROOT)
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ.setdefault("MPLCONFIGDIR", "/tmp/walking-robot-mpl")
sys.path.insert(0, str(ROOT / "tutorials"))
print("Python:", sys.executable)
from importlib.util import find_spec
from shlex import quote
required = ["jax", "yaml", "numpy", "matplotlib", "ipywidgets", "IPython", "walking_robot"]
missing = [name for name in required if find_spec(name) is None]
if missing:
    install = f"{quote(sys.executable)} -m pip install -e {quote(str(ROOT) + '[analysis,notebooks]')}"
    raise ModuleNotFoundError(
        "Missing packages in the selected notebook kernel: " + ", ".join(missing)
        + "\nRun this command in a terminal, then restart the kernel:\n" + install
    )
import jax
jax.config.update("jax_enable_x64", True)
import yaml
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Image
from _widgets import animation_view, preview_design, trajectory_view, show_checks
print("Repository:", ROOT)
print("Simulation device:", jax.devices())


## 2. Change the target speed and inspect the cost

For a predicted mean speed v and target v_target, use
`L = (v - v_target)**2`. Move the target slider to see how the objective changes.
The slope `dL/dv = 2*(v - v_target)` indicates how speed affects the loss.

The design gradient additionally needs NeRD's speed sensitivity to each design
parameter: `dL/ddesign = (dL/dv) * (dv/ddesign)`. The curve below illustrates the
cost definition; it does not run the learned model or establish that a target
speed is physically attainable.


In [ ]:
def speed_cost(mean_speed, target_speed):
    return (np.asarray(mean_speed) - target_speed) ** 2

target = widgets.FloatSlider(value=1, min=.5, max=1.4, step=.05, description="Target (m/s)",
                             continuous_update=False, style={"description_width": "initial"})
probe = widgets.FloatSlider(value=.8, min=.3, max=1.5, step=.05, description="Speed (m/s)",
                            continuous_update=False, style={"description_width": "initial"})

def plot_cost(target_speed, mean_speed):
    speeds = np.linspace(.3, 1.5, 300)
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(speeds, speed_cost(speeds, target_speed), label="Squared speed error")
    ax.axvline(target_speed, color="gray", linestyle="--", label="Target")
    ax.scatter([mean_speed], [speed_cost(mean_speed, target_speed)], color="tab:orange", zorder=3)
    ax.set(xlabel="Mean speed (m/s)", ylabel="Loss ((m/s)²)")
    ax.legend()
    ax.grid(alpha=.2)
    display(fig)
    plt.close(fig)
    print(f"Loss: {float(speed_cost(mean_speed, target_speed)):.4f}; dL/dv: {2*(mean_speed-target_speed):.4f}")

cost_output = widgets.interactive_output(plot_cost, {"target_speed": target, "mean_speed": probe})
display(widgets.HBox([target, probe]), cost_output)


## 3. Compare predictions with solver verification

The values below are recorded results from the expanded study, rather than a
new inference run. One proposed design reached 0.977 m/s after a 1.000 m/s neural
prediction. Another reached only 0.572 m/s after a 1.069 m/s prediction and was
rejected. Change the target above to compare their predicted and verified losses.

The 1.005 m/s candidate is a separate design found by physical solver search.
Changing the loss changes the objective; improving predictions across a broader
design domain requires appropriate training coverage and validation.


In [ ]:
recorded_predictions = np.array([1.000, 1.069411])
recorded_solver_speeds = np.array([.977482, .571520])

def compare_proposals(target_speed):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3))
    x = np.arange(2)
    for ax, predicted, verified, label in [
        (axes[0], recorded_predictions, recorded_solver_speeds, "Mean speed (m/s)"),
        (axes[1], speed_cost(recorded_predictions, target_speed), speed_cost(recorded_solver_speeds, target_speed), "Loss ((m/s)²)"),
    ]:
        ax.bar(x-.18, predicted, width=.36, label="NeRD prediction")
        ax.bar(x+.18, verified, width=.36, label="Solver verification")
        ax.set_xticks(x, ["Proposal A", "Proposal B (rejected)"])
        ax.set_ylabel(label)
        ax.legend()
    axes[0].axhline(target_speed, linestyle="--", color="gray")
    fig.tight_layout()
    display(fig)
    plt.close(fig)

proposal_output = widgets.interactive_output(compare_proposals, {"target_speed": target})
display(proposal_output)


## 4. Inspect the physical candidate and local evidence

The YAML below describes the separate 1.005 m/s physical-search candidate.
Its documented result uses seconds 1–2 of a two-second run and checks two
resolutions and four initial phases. The drawing shows its supported initial
pose. Use its YAML with the simulation workflow to reproduce its motion.

When local training and verification reports exist, this cell also prints their
results. Those generated reports and model checkpoints remain under `outputs/`.


In [ ]:
from walking_robot.expanded_design import ExpandedRobotDesign
configuration = yaml.safe_load(Path("examples/designs/fixed_actuation_1m_s.yaml").read_text())
print(yaml.safe_dump(configuration["design"], sort_keys=False))
preview_design(ExpandedRobotDesign(**configuration["design"]))
TRAINING_REPORT = Path("outputs/expanded_nerd/training/report.json")
VERIFICATION_REPORT = Path("outputs/expanded_nerd/training/physical_verification/report.json")
if TRAINING_REPORT.exists():
    training = json.loads(TRAINING_REPORT.read_text())
    print("Local training report:", TRAINING_REPORT)
    print("Selected iteration:", training.get("selected_iteration"))
    print("Validation trajectory checks passed:", training["validation"]["trajectory_gate_passed"])
    print("Held-out test trajectory checks passed:", training["test"]["trajectory_gate_passed"])
else:
    print("Recorded study: docs/expanded_design.md. Local training report has not been generated.")
if VERIFICATION_REPORT.exists():
    verification = json.loads(VERIFICATION_REPORT.read_text())
    print("Local verification report:", VERIFICATION_REPORT)
    for row in verification.get("proposals", []):
        proposal = row["proposal"]
        print("Target:", proposal["target_m_s"], "m/s · predicted:", proposal["neural_speed_m_s"],
              "m/s · solver:", row["physical_speed_m_s"], "m/s")
        print("Forecast check passed:", row["neural_forecast_passed"], "· target met:", row["target_met"])


## 5. Reproduce training and physical verification

Read [the expanded protocol](../docs/expanded_design.md) before launching this
step: it generates a dataset, trains the model and verifies proposals, and takes
longer than the plotting steps above. Set the official checkout path and
`RUN_TRAINING = True` to execute the commands.

Validation must establish prediction accuracy on independent designs, useful
design sensitivities, and physical feasibility at finer timesteps. A low neural
loss alone does not establish a successful design.


In [ ]:
RUN_TRAINING = False
NERD_ROOT = Path("../neural-robot-dynamics")
commands = [
    [sys.executable, "tools/generate_expanded_data.py"],
    [sys.executable, "tools/generate_expanded_data.py", "--collect"],
    [sys.executable, "tools/train_expanded_nerd.py", "--nerd-root", str(NERD_ROOT)],
    [sys.executable, "tools/verify_expanded_nerd.py"],
    [sys.executable, "tools/verify_expanded_nerd.py", "--collect"],
]
if RUN_TRAINING:
    assert NERD_ROOT.exists(), "Set NERD_ROOT to the official NeRD checkout."
    for command in commands:
        subprocess.run(command, check=True)
else:
    for command in commands:
        print(" ".join(command))
